# Phase 6 — pembanding Opsi A/B untuk resolusi unit dan episode

**REQUIREMENT:** tes sintetis pengelompokan komponen dan pemisahan episode →
replay target TOR3-PAGI → perbandingan keputusan per passage. Rider + motorcycle
merupakan satu motorcycle; pengukuran BC kendaraan tetap berasal dari bbox
mentah kendaraan, bukan bbox rider atau gabungan pasangan.

**ENGINEERING EXPERIMENT:** Opsi B memakai relasi spasial/temporal yang eksplisit,
reset episode pada perubahan unit-family/gap, serta geometri G3 pada episode
terukur yang baru. Ia mempertahankan diagnostic A lama. A original dan A updated
(dengan feedback pengguna) dibaca sebagai snapshot tersimpan; notebook ini
**tidak menjalankan ulang A**.

Dua replay B dijalankan secara eksplisit: `B_machine` dengan grouping dan
kontrol `B_episode_only_no_grouping` dengan reset episode yang sama tetapi
grouping dimatikan. Output dan namespace kontrol disimpan terpisah; ini
membedakan pengaruh episode reset dari pengaruh resolusi hubungan.

Ini replay CPU tanpa video decode, YOLO, BoT-SORT atau encoder ReID. Cache
ReID-on tetap sumber track; embeddings per observation tidak tersedia dan tidak
berpura-pura digunakan. Tidak memulai Phase 7, UI, Docker atau refactor produksi.

**Batas:** accepted mesin B adalah keputusan eksperimen pada scope yang
tercantum dalam konfigurasi. Lebih sedikit pending/banyak accepted belum
membuktikan perbaikan. Tidak ada formal ground truth atau klaim accuracy,
precision, recall, F1 maupun MAPE. Audit salah merge, duplicate/missed passage,
kategori dan sumber crossing harus dibaca terpisah dari reconciliation software.

## Parameter input dan snapshot pembanding

Jalankan dari checkout repository lengkap. Di Kaggle, set `REPO_ROOT` ke path
checkout yang aktual; GPU/video sumber tidak diperlukan untuk replay ini.
`A_UPDATED_RUN_DIR` wajib menunjuk snapshot A yang sudah dibuat dengan feedback
pengguna. `CALIBRATION_PATH` wajib menunjuk bukti pemilihan parameter eksperimen.
Jika path belum diisi, notebook berhenti tanpa mengganti input dengan asumsi.

Konfigurasi B menyatakan scope track target dan parameter association; angka
tersebut adalah kontrol eksperimen, bukan default produk. Mengganti video
memerlukan konfigurasi, source identity, review geometri dan bukti kalibrasi baru.
Restart kernel dan Run All agar source terkini diuji sebelum input aktual dibaca.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'AGENTS.md').is_file() and (REPO_ROOT.parent / 'AGENTS.md').is_file():
    REPO_ROOT = REPO_ROOT.parent

RECORDS_PATH = REPO_ROOT / (
    'docs/evidence/phase_4/20261011T095924_205716Z_90cf7319/'
    'reid_on/track_records.jsonl.gz'
)
GEOMETRY_PATH = REPO_ROOT / 'docs/evidence/phase_5/accepted_geometry/TOR3-PAGI-G3.json'
CONFIG_PATH = REPO_ROOT / 'docs/evidence/phase_6/config/TOR3-PAGI_option_b_v1.json'
A_ORIGINAL_RUN_DIR = REPO_ROOT / 'docs/evidence/phase_6/20261012T144625_030645Z'
A_UPDATED_RUN_DIR = REPO_ROOT / 'docs/evidence/phase_6/option_b_a_updated_20261013'
CALIBRATION_PATH = REPO_ROOT / 'docs/evidence/phase_6/option_b_calibration/frozen_thresholds.json'
OUTPUT_DIR = (
    Path('/kaggle/working/phase_6_option_b')
    if Path('/kaggle/working').is_dir()
    else Path('/tmp/phase_6_option_b')
)
REVIEW_SAMPLE_INDEX_PATH = (
    REPO_ROOT / 'docs/evidence/phase_6/review_samples/TOR3-PAGI/sample_index.json'
)
DISPLAY_REVIEW_TRACK_IDS = (1, 973, 1774, 1323)

## Lingkungan dan kode sumber

Sel ini mengimpor kode dan memeriksa keberadaan path. Konten cache, konfigurasi,
kalibrasi dan snapshot A aktual baru dibaca setelah guard tes sintetis lulus.
Credential dan environment secrets tidak diperiksa atau dicetak.

In [ ]:
import sys
import os
import importlib
from copy import deepcopy
import json
import gzip
import hashlib
import platform
import time
from datetime import datetime, timezone
from collections import Counter

REPO_ROOT = Path(REPO_ROOT).resolve()
HELPER_DIR = REPO_ROOT / 'notebooks'
RECORDS_PATH = Path(RECORDS_PATH).resolve()
GEOMETRY_PATH = Path(GEOMETRY_PATH).resolve()
CONFIG_PATH = Path(CONFIG_PATH).resolve()
A_ORIGINAL_RUN_DIR = Path(A_ORIGINAL_RUN_DIR).resolve()
A_UPDATED_RUN_DIR = None if A_UPDATED_RUN_DIR is None else Path(A_UPDATED_RUN_DIR).resolve()
CALIBRATION_PATH = None if CALIBRATION_PATH is None else Path(CALIBRATION_PATH).resolve()
OUTPUT_DIR = Path(OUTPUT_DIR).resolve()

required_code = {
    'geometry helper': HELPER_DIR / 'phase_5_geometry_helpers.py',
    'counting validation helper': HELPER_DIR / 'phase_6_counting_helpers.py',
    'identity-resolution helper': HELPER_DIR / 'phase_6_identity_resolution_helpers.py',
    'identity-resolution tests': HELPER_DIR / 'phase_6_identity_resolution_tests.py',
}
missing = {name: str(path) for name, path in required_code.items() if not path.is_file()}
if missing:
    raise FileNotFoundError('Source checkout tidak lengkap: ' + json.dumps(missing, indent=2))
sys.path.insert(0, str(HELPER_DIR))
importlib.invalidate_caches()
import phase_5_geometry_helpers as geometry_helpers
import phase_6_counting_helpers as counting_helpers
import phase_6_identity_resolution_helpers as identity_helpers
import phase_6_identity_resolution_tests as identity_tests
geometry_helpers = importlib.reload(geometry_helpers)
counting_helpers = importlib.reload(counting_helpers)
identity_helpers = importlib.reload(identity_helpers)
identity_tests = importlib.reload(identity_tests)
for module in (geometry_helpers, counting_helpers, identity_helpers, identity_tests):
    if Path(module.__file__).resolve().parent != HELPER_DIR.resolve():
        raise RuntimeError('Import berasal dari checkout lain; restart kernel dan Run All.')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, payload):
    Path(path).write_text(json.dumps(
        payload, ensure_ascii=False, allow_nan=False, sort_keys=True, indent=2
    ) + '\n', encoding='utf-8')

def write_jsonl_gzip(path, rows):
    with Path(path).open('wb') as output:
        with gzip.GzipFile(filename='', mode='wb', fileobj=output, mtime=0) as compressed:
            for row in rows:
                compressed.write((json.dumps(
                    row, ensure_ascii=False, allow_nan=False, sort_keys=True
                ) + '\n').encode('utf-8'))

def write_json_gzip(path, payload):
    with Path(path).open('wb') as output:
        with gzip.GzipFile(filename='', mode='wb', fileobj=output, mtime=0) as compressed:
            compressed.write((json.dumps(
                payload, ensure_ascii=False, allow_nan=False, sort_keys=True
            ) + '\n').encode('utf-8'))

def read_jsonl_gzip(path):
    with gzip.open(path, 'rt', encoding='utf-8') as source:
        return [json.loads(line) for line in source if line.strip()]

runtime = {
    'python': sys.version,
    'python_executable': sys.executable,
    'platform': platform.platform(),
    'available_cpu_count': os.cpu_count(),
    'execution_mode': 'CPU_ONLY_RETAINED_TRACK_UNIT_EPISODE_RESOLUTION',
    'gpu_required': False,
    'video_decode_executed': False,
    'detector_tracker_encoder_rerun': False,
    'new_episode_geometry_replay': True,
    'historical_diagnostics_overwritten': False,
    'appearance_vectors_available_or_used': False,
}
print(json.dumps(runtime, indent=2))

## Guard tes sintetis sebelum replay target aktual

Fixtures positif/negatif menguji grouping, reset episode, raw-source preservation,
counting gates dan determinisme. Fixtures bukan observasi video/ground truth.
Kegagalan menghentikan replay; hasil tes dan source hash tetap tersimpan.

In [ ]:
run_started_at = datetime.now(timezone.utc).isoformat()
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = OUTPUT_DIR / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
source_paths = {
    'geometry_helper': Path(geometry_helpers.__file__),
    'counting_validation_helper': Path(counting_helpers.__file__),
    'identity_resolution_helper': Path(identity_helpers.__file__),
    'identity_resolution_tests': Path(identity_tests.__file__),
    'repository_notebook_source': HELPER_DIR / '06b_identity_resolution_comparison.ipynb',
}
source_code_sha256 = {name: sha256_file(path) for name, path in source_paths.items()}
write_json(RUN_DIR / 'runtime.json', runtime)
write_json(RUN_DIR / 'source_code_sha256.json', source_code_sha256)
test_result = identity_tests.run_tests()
write_json(RUN_DIR / 'synthetic_test_results.json', test_result)
print(json.dumps({key: test_result.get(key) for key in (
    'successful', 'tests_run', 'tests_passed', 'failures', 'errors', 'skipped'
)}, indent=2))
if not test_result['successful']:
    raise AssertionError('Guard tes gagal; B tidak dijalankan. Bukti: ' + str(RUN_DIR))
print('Guard lulus; output snapshot:', RUN_DIR)

## Binding konfigurasi, kalibrasi dan snapshot A

File A tetap immutable. A original dan A updated menggunakan review manusia/AI;
B machine tidak menerima review tersebut sebagai override relasi atau kategori.
Kesetaraan video/cache/geometri diperiksa sebelum comparison. Total A full scope
jangan dibandingkan langsung dengan B targeted scope.

In [ ]:
if A_UPDATED_RUN_DIR is None or CALIBRATION_PATH is None:
    raise ValueError('Isi A_UPDATED_RUN_DIR dan CALIBRATION_PATH dengan evidence aktual sebelum replay.')
required_inputs = {
    'records': RECORDS_PATH, 'geometry': GEOMETRY_PATH,
    'configuration': CONFIG_PATH, 'calibration': CALIBRATION_PATH,
}
for label, directory in (('A_original', A_ORIGINAL_RUN_DIR), ('A_updated', A_UPDATED_RUN_DIR)):
    for name in ('report.json', 'aggregate.json', 'accepted_events.jsonl.gz', 'resolved_decisions.jsonl.gz'):
        required_inputs[label + '/' + name] = directory / name
missing = {name: str(path) for name, path in required_inputs.items() if not path.is_file()}
if missing:
    raise FileNotFoundError('Evidence input belum lengkap: ' + json.dumps(missing, indent=2))
input_sha256 = {name: sha256_file(path) for name, path in required_inputs.items()}
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
calibration = json.loads(CALIBRATION_PATH.read_text(encoding='utf-8'))
expected = config['input_identity']
for name in ('records', 'geometry'):
    if input_sha256[name] != expected[name + '_sha256']:
        raise ValueError('Source hash tidak sesuai konfigurasi: ' + name)
if input_sha256['calibration'] != config['calibration_evidence_sha256']:
    raise ValueError('Hash bukti kalibrasi berbeda dari konfigurasi.')
if CALIBRATION_PATH != (REPO_ROOT / config['calibration_evidence_path']).resolve():
    raise ValueError('Path bukti kalibrasi berbeda dari konfigurasi.')
if calibration['provenance']['source_records_sha256'] != expected['records_sha256']:
    raise ValueError('Kalibrasi menggunakan cache sumber berbeda.')
if calibration['thresholds'] != config['association']:
    raise ValueError('Parameter association berbeda dari kontrol yang dibekukan.')
if config.get('policy_version') != 'phase6_option_b_v1' or config.get('metric') != 'passage':
    raise ValueError('Notebook ini khusus eksperimen passage phase6_option_b_v1.')
comparison_track_ids = config.get('comparison_track_ids')
if not isinstance(comparison_track_ids, list) or not comparison_track_ids:
    raise ValueError('Config harus mendeklarasikan comparison_track_ids targeted scope.')
if any(type(track_id) is not int or track_id < 0 for track_id in comparison_track_ids):
    raise ValueError('Scope track ID harus integer nonnegative.')
comparison_track_ids = set(comparison_track_ids)

# Byte-preserving inputs; calibration adalah evidence pemilihan kontrol, bukan benchmark.
(RUN_DIR / 'configuration_input.json').write_bytes(CONFIG_PATH.read_bytes())
(RUN_DIR / 'calibration_input.json').write_bytes(CALIBRATION_PATH.read_bytes())

def validate_aggregate(events, aggregate):
    ids = [event['event_id'] for event in events]
    if len(ids) != len(set(ids)):
        raise AssertionError('Duplicate accepted event ID.')
    if aggregate['total'] != len(events):
        raise AssertionError('Total aggregate berbeda dari jumlah accepted event unik.')
    for key, field in (('by_category', 'category'), ('by_direction', 'direction')):
        actual = Counter(event[field] for event in events)
        if set(actual) - set(aggregate[key]):
            raise AssertionError('Aggregate kehilangan key: ' + key)
        if any(value != actual.get(label, 0) for label, value in aggregate[key].items()):
            raise AssertionError('Aggregate tidak reconcile: ' + key)
    if sorted(aggregate['event_ids']) != sorted(ids):
        raise AssertionError('Aggregate references tidak sama dengan accepted events.')

baseline_snapshots = {}
for label, directory in (('A_original', A_ORIGINAL_RUN_DIR), ('A_updated', A_UPDATED_RUN_DIR)):
    report = json.loads((directory / 'report.json').read_text(encoding='utf-8'))
    identity = report['effective_configuration']['input_identity']
    for key in ('video_sha256', 'records_sha256', 'geometry_sha256', 'variant', 'width', 'height', 'parent_run_id'):
        if identity[key] != expected[key]:
            raise ValueError(label + ' memakai source berbeda: ' + key)
    events = read_jsonl_gzip(directory / 'accepted_events.jsonl.gz')
    aggregate = json.loads((directory / 'aggregate.json').read_text(encoding='utf-8'))
    decisions = read_jsonl_gzip(directory / 'resolved_decisions.jsonl.gz')
    validate_aggregate(events, aggregate)
    scoped_events = [event for event in events if comparison_track_ids.intersection(event['source_track_ids'])]
    scoped_decisions = [row for row in decisions if row['track_id'] in comparison_track_ids]
    baseline_snapshots[label] = {
        'source_directory': str(directory),
        'input_byte_hashes': {name: value for name, value in input_sha256.items() if name.startswith(label + '/')},
        'full_snapshot_aggregate_context_only': aggregate,
        'scoped_accepted_events': scoped_events,
        'scoped_resolved_decisions': scoped_decisions,
        'scoped_status_counts': dict(Counter(row['status'] for row in scoped_decisions)),
        'scoped_unique_accepted_event_count': len(scoped_events),
        'human_review_is_not_automatic_B_feature': True,
    }
write_json_gzip(RUN_DIR / 'a_baseline_comparison_inputs.json.gz', baseline_snapshots)
print(json.dumps({
    'policy_version': config['policy_version'],
    'neutral_band_px_halfwidth': config['neutral_band_px'],
    'confirmation_observations': config['confirmation_observations'],
    'comparison_track_ids': sorted(comparison_track_ids),
    'calibration_sha256': input_sha256['calibration'],
    'A_baselines': {label: {
        'full_snapshot_total_context_only': row['full_snapshot_aggregate_context_only']['total'],
        'target_scope_event_count': row['scoped_unique_accepted_event_count'],
        'target_scope_candidate_statuses': row['scoped_status_counts'],
    } for label, row in baseline_snapshots.items()},
}, indent=2))

## B machine: relasi komponen, vehicle episode dan crossing baru

Resolver menerima cache dan konfigurasi, tanpa review-ID overrides. Family/gap
reset diterapkan generik sebelum geometri episode baru. Person BC tidak boleh
menggantikan vehicle BC. Relasi `COMPONENT_OF` dibedakan dari `SAME_VEHICLE`;
alternative assignment dan konflik yang belum selesai tetap pending.

Machine outputs disimpan terpisah dari A reviewed. Pemeriksaan berikut memastikan
accepted-only totals, bukan correctness identitas fisik pada seluruh video.

In [ ]:
replay_started_at = datetime.now(timezone.utc).isoformat()
clock = time.perf_counter()
result = identity_helpers.run_identity_resolution(
    RECORDS_PATH, GEOMETRY_PATH, config=config,
)
elapsed_seconds = time.perf_counter() - clock
required_result_keys = {
    'episodes', 'relations', 'derived_diagnostics', 'machine_decisions',
    'accepted_events', 'aggregate', 'report', 'source_index',
}
if not required_result_keys.issubset(result):
    raise ValueError('Output helper B tidak memenuhi interface eksperimen.')
accepted_events = result['accepted_events']
for event in accepted_events:
    if (event.get('type') != 'CountEvent' or event.get('status') != 'accepted'
            or event.get('accepted_count_event') is not True):
        raise AssertionError('Accepted output harus typed CountEvent accepted.')
validate_aggregate(accepted_events, result['aggregate'])
for name in ('episodes', 'relations', 'derived_diagnostics', 'machine_decisions', 'accepted_events'):
    write_jsonl_gzip(RUN_DIR / (name + '.jsonl.gz'), result[name])
write_json_gzip(RUN_DIR / 'source_index.json.gz', result['source_index'])
write_json(RUN_DIR / 'aggregate.json', result['aggregate'])
write_json(RUN_DIR / 'software_report.json', result['report'])
report = {
    'run_id': run_id,
    'claim_type': 'EXPERIMENT_RESULT_TARGETED_MACHINE_IDENTITY_EPISODE_RESOLUTION_NOT_TRAFFIC_BENCHMARK',
    'replay_started_at_utc': replay_started_at,
    'replay_completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'measured_replay_elapsed_seconds': elapsed_seconds,
    'runtime': runtime,
    'source_code_sha256': source_code_sha256,
    'input_sha256': input_sha256,
    'effective_configuration': config,
    'calibration_evidence_sha256': input_sha256['calibration'],
    'synthetic_tests_successful': test_result['successful'],
    'synthetic_test_results_sha256': sha256_file(RUN_DIR / 'synthetic_test_results.json'),
    'software_report': result['report'],
    'comparison_track_ids': sorted(comparison_track_ids),
    'new_episode_geometry_replay': True,
    'A_original_or_updated_executed_in_this_notebook': False,
    'manual_feedback_applied_to_B_machine': False,
    'identity_resolver_executions_planned': ['B_machine', 'B_episode_only_no_grouping'],
    'formal_ground_truth_available': False,
    'whole_video_physical_audit_complete': False,
    'selected_suitable_product_parameters': None,
    'source_pts_status': 'UNKNOWN_FRAME_FPS_TIMESTAMPS_ARE_ESTIMATES',
    'phase_7_started': False,
}
write_json(RUN_DIR / 'report.json', report)
print('Replay B CPU selesai:', round(elapsed_seconds, 3), 'detik')
print(json.dumps({'aggregate': result['aggregate'], 'software_report': result['report']}, indent=2))

## Kontrol ablation: episode sama, grouping dimatikan

Replay kedua memakai sumber, G3, numeric controls dan reset episode yang sama.
Hanya `grouping_enabled=False`. Kontrol ini memeriksa kontribusi grouping secara
terpisah dari episode reset; hasilnya tetap subtotal eksperimen. Output berada
di subdirektori `ablation/` dan tidak ditambahkan ke aggregate B machine.


In [ ]:
ABLATION_DIR = RUN_DIR / 'ablation'
ABLATION_DIR.mkdir(parents=True, exist_ok=False)
ablation_config = deepcopy(config)
ablation_config['grouping_enabled'] = False
write_json(ABLATION_DIR / 'effective_configuration.json', ablation_config)
ablation_started_at = datetime.now(timezone.utc).isoformat()
ablation_clock = time.perf_counter()
ablation_result = identity_helpers.run_identity_resolution(
    RECORDS_PATH, GEOMETRY_PATH, config=ablation_config,
)
ablation_elapsed_seconds = time.perf_counter() - ablation_clock
if not required_result_keys.issubset(ablation_result):
    raise ValueError('Output kontrol ablation tidak memenuhi interface eksperimen.')
for event in ablation_result['accepted_events']:
    if (event.get('type') != 'CountEvent' or event.get('status') != 'accepted'
            or event.get('accepted_count_event') is not True):
        raise AssertionError('Output ablation accepted harus typed CountEvent accepted.')
validate_aggregate(ablation_result['accepted_events'], ablation_result['aggregate'])
for name in ('episodes', 'relations', 'derived_diagnostics', 'machine_decisions', 'accepted_events'):
    write_jsonl_gzip(ABLATION_DIR / (name + '.jsonl.gz'), ablation_result[name])
write_json(ABLATION_DIR / 'aggregate.json', ablation_result['aggregate'])
write_json(ABLATION_DIR / 'software_report.json', ablation_result['report'])
write_json(ABLATION_DIR / 'report.json', {
    'run_id': run_id,
    'control': 'B_episode_only_no_grouping',
    'claim_type': 'EXPERIMENT_RESULT_MECHANISM_ABLATION_NOT_MODEL_PERFORMANCE',
    'replay_started_at_utc': ablation_started_at,
    'replay_completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'measured_replay_elapsed_seconds': ablation_elapsed_seconds,
    'source_code_sha256': source_code_sha256,
    'input_sha256': input_sha256,
    'effective_configuration': ablation_config,
    'effective_configuration_sha256': sha256_file(ABLATION_DIR / 'effective_configuration.json'),
    'software_report': ablation_result['report'],
    'manual_feedback_applied': False,
    'detector_tracker_encoder_rerun': False,
    'new_episode_geometry_replay': True,
    'formal_ground_truth_available': False,
})
report['identity_resolver_executions_completed'] = ['B_machine', 'B_episode_only_no_grouping']
report['ablation_control_directory'] = 'ablation'
write_json(RUN_DIR / 'report.json', report)
print('Replay ablation CPU selesai:', round(ablation_elapsed_seconds, 3), 'detik')
print(json.dumps({
    'control': 'B_episode_only_no_grouping',
    'aggregate': ablation_result['aggregate'],
    'software_report': ablation_result['report'],
}, indent=2))


## Inventory comparison per candidate/passage dan antrian audit

Snapshot A dan B memakai namespace event berbeda; equality event-ID lintas opsi
bukan definisi passage yang sama. Inventory menyimpan diagnostic intervals,
source track/episode serta keputusan lengkap agar audit dapat mencocokkan
**unit fisik + episode passage** secara eksplisit.

Status/count mesin berikut belum menilai false merge atau missed passage.
Review sebenarnya harus memeriksa foto/temporal setiap kasus, pasangan negatif,
kategori dan pengukuran crossing. Temuan itu diretain dalam berkas audit terpisah;
tidak diisi oleh notebook dengan klaim visual atau ground truth palsu.

In [ ]:
status_counts = dict(Counter(row['status'] for row in result['machine_decisions']))
pending = [row for row in result['machine_decisions'] if row['status'] == 'pending_review']
rejected = [row for row in result['machine_decisions'] if row['status'] == 'rejected']
write_jsonl_gzip(RUN_DIR / 'pending_decisions.jsonl.gz', pending)
write_jsonl_gzip(RUN_DIR / 'rejected_decisions.jsonl.gz', rejected)
comparison_inventory = {
    'claim_type': 'ACTUAL_SOFTWARE_DECISION_INVENTORY_AWAITING_CASE_PASSAGE_AUDIT',
    'same_source_identity': expected,
    'comparison_track_ids': sorted(comparison_track_ids),
    'A_baselines': baseline_snapshots,
    'B_machine': {
        'machine_decisions': result['machine_decisions'],
        'accepted_events': result['accepted_events'],
        'aggregate': result['aggregate'],
        'relations': result['relations'],
    },
    'B_episode_only_no_grouping': {
        'machine_decisions': ablation_result['machine_decisions'],
        'accepted_events': ablation_result['accepted_events'],
        'aggregate': ablation_result['aggregate'],
        'relations': ablation_result['relations'],
    },
    'event_ids_are_not_cross_policy_passage_matching_keys': True,
    'full_A_vs_targeted_B_total_is_not_a_performance_comparison': True,
    'false_merge_duplicate_category_crossing_source_visual_audit_status': 'NOT_EXECUTED_BY_THIS_NOTEBOOK',
    'lower_pending_alone_establishes_improvement': False,
}
write_json_gzip(RUN_DIR / 'comparison_inventory.json.gz', comparison_inventory)
summary = {
    'B_machine_candidate_status_counts': status_counts,
    'B_machine_unique_accepted_event_subtotal': len(result['accepted_events']),
    'B_machine_aggregate': result['aggregate'],
    'B_episode_only_no_grouping_aggregate': ablation_result['aggregate'],
    'B_episode_only_no_grouping_status_counts': dict(Counter(
        row['status'] for row in ablation_result['machine_decisions']
    )),
    'A_baseline_target_scope_event_counts': {
        label: row['scoped_unique_accepted_event_count'] for label, row in baseline_snapshots.items()
    },
    'interpretation': 'TARGETED_EXPERIMENT_SUBTOTAL_NOT_COMPLETE_VIDEO_TRAFFIC_RESULT',
    'visual_case_audit_required': ['false_merge', 'duplicate_or_missed_passage', 'category', 'vehicle_BC_source'],
}
write_json(RUN_DIR / 'summary.json', summary)
print(json.dumps(summary, indent=2))

## Manifest dan pemeriksaan perubahan input/source

Input maupun source code diperiksa ulang. Run timestamp adalah provenance;
logical episode/relation/event ID harus deterministik dan tidak memakai clock.
Rerun adalah snapshot terpisah, bukan penambahan total dari snapshot sebelumnya.

In [ ]:
for name, path in required_inputs.items():
    if sha256_file(path) != input_sha256[name]:
        raise RuntimeError('Input berubah selama replay: ' + name)
for name, path in source_paths.items():
    if sha256_file(path) != source_code_sha256[name]:
        raise RuntimeError('Kode berubah selama replay: ' + name)
validate_aggregate(result['accepted_events'], result['aggregate'])
validate_aggregate(ablation_result['accepted_events'], ablation_result['aggregate'])
manifest = {
    'run_id': run_id,
    'run_started_at_utc': run_started_at,
    'run_completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'claim_type': 'EXPERIMENT_RESULT_PHASE6_TARGETED_OPTION_B_SOFTWARE_ARTIFACTS',
    'source_code_sha256': source_code_sha256,
    'input_sha256': input_sha256,
    'comparison_track_ids': sorted(comparison_track_ids),
    'accepted_unique_event_subtotal': len(result['accepted_events']),
    'artifacts': [
        {'path': path.relative_to(RUN_DIR).as_posix(), 'size_bytes': path.stat().st_size, 'sha256': sha256_file(path)}
        for path in sorted(RUN_DIR.rglob('*')) if path.is_file()
    ],
}
write_json(RUN_DIR / 'execution_manifest.json', manifest)
print('Snapshot B dan manifest:', RUN_DIR)
print('Lanjutkan audit per passage: salah merge, duplikasi, kategori dan source crossing.')
print('Tidak memulai Phase 7; angka kontrol belum menjadi parameter produksi.')

## Opsional: tampilkan contact sheets historis yang tersedia

Gambar di bawah berasal dari evidence pixel Phase 6 sebelumnya. Menampilkannya
tidak mengesahkan seluruh episode B baru. Kasus yang memerlukan frame baru
(termasuk origin episode baru ID1772) harus memperoleh review sample tambahan
ber-hash pada proses audit terpisah. Sel ini tidak decode video atau membuat
keputusan review otomatis.

In [ ]:
sample_index_path = Path(REVIEW_SAMPLE_INDEX_PATH).resolve()
if not sample_index_path.is_file():
    print('Index foto historis tidak tersedia; tidak ada review visual yang dijalankan.')
else:
    sample_index = json.loads(sample_index_path.read_text(encoding='utf-8'))
    if sample_index.get('source_video_sha256') != expected['video_sha256']:
        raise ValueError('Contact sheets berasal dari source video berbeda.')
    try:
        from IPython.display import Image, display
    except ImportError:
        Image = display = None
    for case in sample_index['cases']:
        if case['track_id'] not in DISPLAY_REVIEW_TRACK_IDS:
            continue
        print('Historical review track', case['track_id'], 'frames:', case['frames'])
        for kind in ('raw', 'overlay'):
            artifact = case['artifacts'][kind]
            image_path = REPO_ROOT / artifact['path']
            if not image_path.is_file() or sha256_file(image_path) != artifact['sha256']:
                raise ValueError('Contact sheet hilang/berubah: ' + str(image_path))
            if display is None:
                print(kind, image_path)
            else:
                print(kind)
                display(Image(filename=str(image_path)))